# Sauti-Ya-Kenya: Swahili TTS Fine-Tuning on SpeechT5

**Model:** `microsoft/speecht5_tts` fine-tuned on Google WaxalNLP Swahili TTS  
**Compute:** Modal (A100-80GB)  
**Tracking:** Weights & Biases  
**Dataset:** `google/WaxalNLP` — `swa_tts` split  

## Notebook structure

1. **Configuration** — Modal environment, hyperparameters, W&B setup  
2. **Swahili text normalizer** — Pre-tokenizer cleaning for Swahili phonology  
3. **Training** — Data loading, preprocessing, speaker embeddings, training loop  
4. **Evaluation** — Speaker similarity (cosine x-vector) and intelligibility (ASR-based WER)  
5. **Inference** — Generate and listen to synthesized Swahili speech  


In [23]:
import os
import modal

modal.enable_output()

# -- Hyperparameters -----------------------------------------------------------
# Adjust these before launching a run. Grouped here for visibility.

CONFIG = {
    # Data
    "dataset": "google/WaxalNLP",
    "dataset_config": "swa_tts",
    "sample_rate": 16_000,
    "min_audio_sec": 0.5,
    "max_audio_sec": 20,
    "min_text_len": 5,
    "max_text_len": 300,
    "val_split": 0.1,
    "seed": 42,

    # Model
    "base_checkpoint": "microsoft/speecht5_tts",
    "vocoder_checkpoint": "microsoft/speecht5_hifigan",
    "speaker_encoder": "speechbrain/spkrec-xvect-voxceleb",

    # Training
    "batch_size": 8,
    "gradient_accumulation_steps": 4,   # effective batch = 8 * 4 = 32
    "learning_rate": 2e-5,
    "warmup_steps": 300,
    "max_steps": 3000,
    "save_steps": 1000,
    "save_total_limit": 3,
    "logging_steps": 50,
    "fp16": True,
    "gradient_checkpointing": True,

    # Eval
    "eval_steps": 500,                  # run eval every N steps
    "eval_samples": 6,                  # number of utterances to synthesize for eval

    # Paths (inside Modal container)
    "output_dir": "/checkpoints/sauti_speecht5_v8",
    "final_model_dir": "/checkpoints/sauti_speecht5_final_v8",

    # W&B
    "wandb_project": "sauti-ya-kenya",
    "wandb_run_name": "speecht5-swa-v8",
}

# -- Modal environment ---------------------------------------------------------
# Pin exact versions for reproducibility. espeak-ng is required by the
# SpeechT5 tokenizer for phoneme fallback; jiwer provides WER computation.

image = (
    modal.Image.debian_slim(python_version="3.12")
    .apt_install("ffmpeg", "git", "espeak-ng", "build-essential")
    .pip_install(
        "torch",
        "torchaudio",
        "transformers",
        "accelerate",
        "datasets==2.19.0",
        "soundfile",
        "librosa",
        "speechbrain",
        "sentencepiece",
        "jiwer",
        "wandb",
        "phonemizer",
    )
)

app = modal.App("sauti-speecht5-v8")
vol = modal.Volume.from_name("sauti-tts-volume", create_if_missing=True)


## Swahili text normalizer

SpeechT5 ships with a SentencePiece BPE tokenizer trained on English text.
Swahili has properties that need explicit handling before tokenization:

- **Prenasalized consonants** — clusters like *mb*, *nd*, *ng'*, *nj* are single
  phonemes in Swahili but get split by an English-trained BPE. We cannot change
  the frozen tokenizer vocabulary, but we can ensure the input text is clean and
  consistent so the model learns stable mappings for these byte sequences.
- **Numbers** — digits should be expanded to Swahili words so the model does not
  have to learn digit-to-speech mapping from scratch on limited data.
- **Unicode** — NFC normalization prevents visually identical but byte-different
  strings from fragmenting the learned representations.
- **Punctuation** — standardize quote styles, dashes, and whitespace.

The normalizer runs *before* the SpeechT5 processor on every text input, both
during training and inference.


In [24]:
# -- Swahili text normalizer ---------------------------------------------------
# This module is defined at the top level so it can be imported inside the Modal
# function. All functions are pure (no side effects, no GPU dependency).

SWAHILI_ONES = [
    "", "moja", "mbili", "tatu", "nne", "tano",
    "sita", "saba", "nane", "tisa"
]
SWAHILI_TENS = [
    "", "kumi", "ishirini", "thelathini", "arobaini", "hamsini",
    "sitini", "sabini", "themanini", "tisini"
]


def _number_to_swahili(n: int) -> str:
    """Convert an integer (0 to ~999 million) to Swahili words.

    Swahili number grammar: power word precedes the count.
        100  = mia moja        (not "moja mia")
        500  = mia tano
        1000 = elfu             (elfu alone implies one thousand)
        1500 = elfu mia tano
        2000 = elfu mbili
    "na" (and) is used only before the final ones digit:
        101  = mia moja na moja
        1501 = elfu mia tano na moja
    """
    if n == 0:
        return "sifuri"
    if n < 0:
        return "hasi " + _number_to_swahili(-n)
    if n < 10:
        return SWAHILI_ONES[n]
    if n < 100:
        tens, ones = divmod(n, 10)
        parts = [SWAHILI_TENS[tens]]
        if ones:
            parts.append("na")
            parts.append(SWAHILI_ONES[ones])
        return " ".join(parts)
    if n < 1000:
        count, remainder = divmod(n, 100)
        # Swahili: mia + count (mia moja, mia mbili, mia tano)
        parts = ["mia", SWAHILI_ONES[count]]
        if remainder:
            if remainder < 10:
                parts.append("na")
            parts.append(_number_to_swahili(remainder))
        return " ".join(parts)
    if n < 1_000_000:
        count, remainder = divmod(n, 1000)
        # Swahili: elfu (+ count if > 1)
        if count == 1:
            parts = ["elfu"]
        else:
            parts = ["elfu", _number_to_swahili(count)]
        if remainder:
            if remainder < 10:
                parts.append("na")
            parts.append(_number_to_swahili(remainder))
        return " ".join(parts)
    if n < 1_000_000_000:
        count, remainder = divmod(n, 1_000_000)
        if count == 1:
            parts = ["milioni"]
        else:
            parts = ["milioni", _number_to_swahili(count)]
        if remainder:
            parts.append(_number_to_swahili(remainder))
        return " ".join(parts)
    # Fallback: read digits individually
    return " ".join(SWAHILI_ONES[int(d)] if d != "0" else "sifuri" for d in str(n))


def normalize_swahili_text(text: str) -> str:
    """Clean and normalize Swahili text for SpeechT5 input.

    Steps:
        1. Unicode NFC normalization
        2. Number-to-word expansion
        3. Punctuation standardization
        4. Whitespace collapse
    """
    import unicodedata
    import re

    # 1. Unicode normalization (NFC)
    text = unicodedata.normalize("NFC", text)

    # 2. Expand numbers to Swahili words
    #    Matches integers (with optional comma grouping like 1,000)
    def _expand_number(match):
        raw = match.group(0).replace(",", "")
        try:
            return _number_to_swahili(int(raw))
        except (ValueError, RecursionError):
            return raw

    text = re.sub(r"\b[\d,]+\b", _expand_number, text)

    # 3. Standardize punctuation
    text = text.replace("\u2018", "'").replace("\u2019", "'")   # curly quotes
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")   # dashes
    text = re.sub(r"[\u2026]", "...", text)                     # ellipsis

    # 4. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Quick sanity checks
assert normalize_swahili_text("Kuna watu 1,500 hapa") == "Kuna watu elfu mia tano hapa"
assert normalize_swahili_text("Siku ya 3") == "Siku ya tatu"
assert normalize_swahili_text("Watu 42 walikuja") == "Watu arobaini na mbili walikuja"
assert normalize_swahili_text("Mwaka 2000") == "Mwaka elfu mbili"
print("Normalizer OK")

Normalizer OK


In [25]:
@app.function(
    image=image,
    gpu="RTX-PRO-6000",
    timeout=60 * 60 * 6,       # 6 hours max
    volumes={"/checkpoints": vol},
    secrets=[modal.Secret.from_name("wandb-secret")],   # WANDB_API_KEY
)
def train():
    """Fine-tune SpeechT5 on WaxalNLP Swahili TTS data.

    This function runs entirely inside the Modal container. It:
        1. Loads and filters the WaxalNLP swa_tts dataset
        2. Normalizes Swahili text
        3. Extracts x-vector speaker embeddings per sample
        4. Trains with Seq2SeqTrainer and logs to W&B
        5. Runs evaluation (speaker similarity + WER) at intervals
        6. Saves final checkpoint to persistent volume
    """
    import torch
    import numpy as np
    import wandb
    import soundfile as sf
    from collections import Counter
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union

    from datasets import load_dataset, Audio
    from transformers import (
        SpeechT5Processor,
        SpeechT5ForTextToSpeech,
        SpeechT5HifiGan,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
    )
    from speechbrain.inference.speaker import EncoderClassifier

    C = CONFIG  # shorthand

    # -- W&B init --------------------------------------------------------------
    wandb.init(
        project=C["wandb_project"],
        name=C["wandb_run_name"],
        config=C,
    )

    print("[1/7] Loading dataset: %s (%s)" % (C["dataset"], C["dataset_config"]))
    ds = load_dataset(C["dataset"], C["dataset_config"], split="train")
    ds = ds.cast_column("audio", Audio(sampling_rate=C["sample_rate"]))

    # Log speaker distribution to W&B for reference
    speaker_counts = Counter(ds["speaker_id"])
    print("  Speaker distribution: %s" % speaker_counts.most_common())
    print("  Total raw samples: %d" % len(ds))
    wandb.log({"data/total_raw_samples": len(ds)})

    # -- Filter ----------------------------------------------------------------
    print("[2/7] Filtering dataset (length and quality constraints)")

    def is_valid(example):
        text_len = len(example["text"].strip())
        audio_len = len(example["audio"]["array"])
        sr = C["sample_rate"]
        return (
            C["min_text_len"] < text_len < C["max_text_len"]
            and sr * C["min_audio_sec"] < audio_len < sr * C["max_audio_sec"]
        )

    ds = ds.filter(is_valid)
    print("  Samples after filtering: %d" % len(ds))
    wandb.log({"data/filtered_samples": len(ds)})

    # -- Normalize text --------------------------------------------------------
    print("[3/7] Normalizing Swahili text")

    def normalize_text(example):
        example["text"] = normalize_swahili_text(example["text"])
        return example

    ds = ds.map(normalize_text)

    # -- Load models -----------------------------------------------------------
    print("[4/7] Loading SpeechT5 + SpeechBrain speaker encoder")
    processor = SpeechT5Processor.from_pretrained(C["base_checkpoint"])
    model = SpeechT5ForTextToSpeech.from_pretrained(C["base_checkpoint"])
    model.config.use_cache = False  # required for gradient checkpointing

    vocoder = SpeechT5HifiGan.from_pretrained(C["vocoder_checkpoint"]).to("cuda")

    spk_model = EncoderClassifier.from_hparams(
        source=C["speaker_encoder"],
        savedir="/tmp/speechbrain",
        run_opts={"device": "cuda"},
    )

    # -- Preprocessing ---------------------------------------------------------
    # NOTE: num_proc is intentionally omitted. SpeechBrain's CUDA tensors are
    # not picklable, so parallel map workers would fail. Sequential processing
    # takes ~10-15 min for ~1300 samples on A100; acceptable for this data size.

    print("[5/7] Preprocessing (text -> tokens, audio -> spectrograms, speaker embeddings)")

    def extract_speaker_embedding(waveform):
        """Compute a normalized x-vector speaker embedding from raw audio."""
        with torch.no_grad():
            wav = torch.tensor(waveform, dtype=torch.float32).unsqueeze(0).to(spk_model.device)
            embedding = spk_model.encode_batch(wav)
            embedding = torch.nn.functional.normalize(embedding, dim=2)
            return embedding.squeeze().cpu().numpy()

    def prepare_example(example):
        """Convert one dataset row into model-ready tensors.

        Returns:
            input_ids:          Tokenized text sequence
            labels:             Log-mel spectrogram frames
            speaker_embeddings: 512-dim x-vector
        """
        audio = example["audio"]
        processed = processor(
            text=example["text"],
            audio_target=audio["array"],
            sampling_rate=audio["sampling_rate"],
            return_attention_mask=False,
        )
        processed["labels"] = processed["labels"][0]
        processed["speaker_embeddings"] = extract_speaker_embedding(audio["array"])
        return processed

    ds = ds.map(prepare_example, remove_columns=ds.column_names)
    print("  Processed samples: %d" % len(ds))

    # -- Train / validation split ----------------------------------------------
    ds = ds.train_test_split(test_size=C["val_split"], seed=C["seed"])
    print("  Train: %d | Validation: %d" % (len(ds["train"]), len(ds["test"])))
    wandb.log({
        "data/train_samples": len(ds["train"]),
        "data/val_samples": len(ds["test"]),
    })

    # -- Data collator ---------------------------------------------------------
    @dataclass
    class TTSCollator:
        """Pad input_ids and spectrogram labels to uniform length within a batch.

        Handles SpeechT5's reduction_factor alignment requirement and masks
        padded label frames with -100 so they are ignored by the loss.
        """
        processor: Any

        def __call__(
            self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
        ) -> Dict[str, torch.Tensor]:
            input_ids = [{"input_ids": f["input_ids"]} for f in features]
            label_features = [{"input_values": f["labels"]} for f in features]
            speaker_features = [f["speaker_embeddings"] for f in features]

            batch = self.processor.pad(
                input_ids=input_ids,
                labels=label_features,
                return_tensors="pt",
            )

            # Mask padded spectrogram frames
            batch["labels"] = batch["labels"].masked_fill(
                batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100
            )
            del batch["decoder_attention_mask"]

            # Align to reduction_factor (SpeechT5 default = 2)
            if model.config.reduction_factor > 1:
                target_lengths = torch.tensor(
                    [len(f["input_values"]) for f in label_features]
                )
                target_lengths = target_lengths.new(
                    [l - l % model.config.reduction_factor for l in target_lengths]
                )
                max_len = max(target_lengths)
                batch["labels"] = batch["labels"][:, :max_len]

            batch["speaker_embeddings"] = torch.tensor(speaker_features)
            return batch

    collator = TTSCollator(processor=processor)

    # -- Training --------------------------------------------------------------
    print("[6/7] Starting fine-tuning (%d steps)" % C["max_steps"])

    training_args = Seq2SeqTrainingArguments(
        output_dir=C["output_dir"],
        per_device_train_batch_size=C["batch_size"],
        gradient_accumulation_steps=C["gradient_accumulation_steps"],
        learning_rate=C["learning_rate"],
        warmup_steps=C["warmup_steps"],
        max_steps=C["max_steps"],
        gradient_checkpointing=C["gradient_checkpointing"],
        fp16=C["fp16"],
        eval_strategy="steps",
        eval_steps=C["eval_steps"],
        save_steps=C["save_steps"],
        save_total_limit=C["save_total_limit"],
        logging_steps=C["logging_steps"],
        report_to="wandb",
        load_best_model_at_end=False,
        # Disable predict_with_generate to avoid eval_loss KeyError.
        # We run custom eval (speaker sim + WER) separately below.
        predict_with_generate=False,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        data_collator=collator,
        processing_class=processor,
    )

    trainer.train()

    # -- Save final model ------------------------------------------------------
    print("[7/7] Saving final model and running evaluation")
    trainer.save_model(C["final_model_dir"])

    # -- Post-training evaluation: generate test samples -----------------------
    # Synthesize a fixed set of phrases and log audio + metrics to W&B.
    model.eval()
    test_texts = [
        "Habari yako? Ninatumai una siku njema leo.",
        "Asante sana kwa msaada wako.",
        "Teknolojia ya akili bandia itabadilisha maisha yetu barani Afrika.",
        "Tafadhali niambie habari zaidi kuhusu huduma hii.",
        "Ninahitaji msaada wa kukokotoa mapato yangu.",
        "Karibu kwenye mfumo wetu mpya.",
    ]

    # Use first training sample's speaker embedding as reference
    ref_embedding = torch.tensor(ds["train"][0]["speaker_embeddings"]).unsqueeze(0)

    audio_table = wandb.Table(columns=["text", "audio", "duration_sec"])
    for i, text in enumerate(test_texts):
        normalized = normalize_swahili_text(text)
        inputs = processor(text=normalized, return_tensors="pt")
        with torch.no_grad():
            speech = model.generate_speech(
                inputs["input_ids"].to("cuda"),
                ref_embedding.to("cuda"),
                vocoder=vocoder,
            )
        wav = speech.cpu().numpy()
        duration = len(wav) / C["sample_rate"]

        # Save to volume
        out_path = "/checkpoints/test_v8_%d.wav" % (i + 1)
        sf.write(out_path, wav, C["sample_rate"])

        # Log to W&B
        audio_table.add_data(
            text,
            wandb.Audio(wav, sample_rate=C["sample_rate"]),
            round(duration, 2),
        )
        print("  Generated: test_v8_%d.wav (%.1fs)" % (i + 1, duration))

    wandb.log({"eval/generated_samples": audio_table})

    vol.commit()
    wandb.finish()
    print("Done. Model and %d test samples saved to volume." % len(test_texts))


# -- Launch --------------------------------------------------------------------
if __name__ == "__main__":
    with app.run():
        train.remote()


## Evaluation

Two automated metrics, designed to run without human annotators:

1. **Speaker similarity** — Cosine similarity between x-vector embeddings of
   synthesized audio and reference audio from the same speaker. Measures whether
   the generated voice *sounds like* the target speaker.
2. **Intelligibility (WER)** — Run a Swahili-capable ASR model (Whisper small)
   on the synthesized audio, then compute Word Error Rate against the input text.
   Measures whether the output is *understandable*.

These complement the audio samples logged to W&B during training.


In [26]:
@app.function(
    image=image,
    gpu="RTX-PRO-6000",
    timeout=60 * 60 * 2,
    volumes={"/checkpoints": vol},
    secrets=[modal.Secret.from_name("wandb-secret")],
)
def evaluate():
    """Run speaker similarity and WER evaluation on saved test samples.

    Requires: test WAV files and the fine-tuned model on the volume
    (produced by the train function).
    """
    import torch
    import numpy as np
    import soundfile as sf
    import wandb
    from jiwer import wer
    from transformers import (
        SpeechT5Processor,
        SpeechT5ForTextToSpeech,
        SpeechT5HifiGan,
        WhisperProcessor,
        WhisperForConditionalGeneration,
    )
    from speechbrain.inference.speaker import EncoderClassifier
    from datasets import load_dataset, Audio

    C = CONFIG

    wandb.init(
        project=C["wandb_project"],
        name=C["wandb_run_name"] + "-eval",
        config=C,
    )

    # -- Load models -----------------------------------------------------------
    print("[eval] Loading models")
    processor = SpeechT5Processor.from_pretrained(C["base_checkpoint"])
    model = SpeechT5ForTextToSpeech.from_pretrained(C["final_model_dir"])
    model.eval().to("cuda")
    vocoder = SpeechT5HifiGan.from_pretrained(C["vocoder_checkpoint"]).to("cuda")

    spk_model = EncoderClassifier.from_hparams(
        source=C["speaker_encoder"],
        savedir="/tmp/speechbrain",
        run_opts={"device": "cuda"},
    )

    # Whisper small has multilingual support including Swahili
    print("[eval] Loading Whisper small for ASR-based WER")
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-small"
    ).to("cuda")
    whisper_model.eval()

    # -- Reference speaker embedding -------------------------------------------
    print("[eval] Extracting reference speaker embedding")
    ds = load_dataset(C["dataset"], C["dataset_config"], split="train", streaming=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=C["sample_rate"]))
    ref_sample = next(iter(ds.filter(lambda x: x["speaker_id"] == "6")))

    with torch.no_grad():
        ref_wav = torch.tensor(
            ref_sample["audio"]["array"], dtype=torch.float32
        ).unsqueeze(0).to("cuda")
        ref_embed = spk_model.encode_batch(ref_wav)
        ref_embed = torch.nn.functional.normalize(ref_embed, dim=2).squeeze()

    # -- Evaluation loop -------------------------------------------------------
    eval_texts = [
        "Habari yako? Ninatumai una siku njema leo.",
        "Asante sana kwa msaada wako.",
        "Teknolojia ya akili bandia itabadilisha maisha yetu barani Afrika.",
        "Tafadhali niambie habari zaidi kuhusu huduma hii.",
        "Ninahitaji msaada wa kukokotoa mapato yangu.",
        "Karibu kwenye mfumo wetu mpya.",
    ]

    speaker_embedding = torch.nn.functional.normalize(ref_embed, dim=0).unsqueeze(0)

    similarities = []
    wer_scores = []

    results_table = wandb.Table(
        columns=["text", "audio", "speaker_sim", "wer", "asr_transcript"]
    )

    for text in eval_texts:
        normalized = normalize_swahili_text(text)
        inputs = processor(text=normalized, return_tensors="pt")

        # Synthesize
        with torch.no_grad():
            speech = model.generate_speech(
                inputs["input_ids"].to("cuda"),
                speaker_embedding.to("cuda"),
                vocoder=vocoder,
            )
        wav = speech.cpu().numpy()

        # 1. Speaker similarity (cosine of x-vectors)
        with torch.no_grad():
            gen_wav = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to("cuda")
            gen_embed = spk_model.encode_batch(gen_wav)
            gen_embed = torch.nn.functional.normalize(gen_embed, dim=2).squeeze()
            cos_sim = torch.nn.functional.cosine_similarity(
                ref_embed.unsqueeze(0), gen_embed.unsqueeze(0)
            ).item()
        similarities.append(cos_sim)

        # 2. ASR-based WER (Whisper transcription)
        whisper_input = whisper_processor(
            wav, sampling_rate=C["sample_rate"], return_tensors="pt"
        ).input_features.to("cuda")
        forced_ids = whisper_processor.get_decoder_prompt_ids(
            language="sw", task="transcribe"
        )
        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                whisper_input, forced_decoder_ids=forced_ids
            )
        transcript = whisper_processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()
        word_error = wer(text.lower(), transcript.lower())
        wer_scores.append(word_error)

        results_table.add_data(
            text,
            wandb.Audio(wav, sample_rate=C["sample_rate"]),
            round(cos_sim, 4),
            round(word_error, 4),
            transcript,
        )
        print("  [%.3f sim | %.3f WER] %s" % (cos_sim, word_error, text[:50]))

    # -- Aggregate metrics -----------------------------------------------------
    mean_sim = np.mean(similarities)
    mean_wer = np.mean(wer_scores)
    print("\n  Mean speaker similarity: %.4f" % mean_sim)
    print("  Mean WER: %.4f" % mean_wer)

    wandb.log({
        "eval/mean_speaker_similarity": mean_sim,
        "eval/mean_wer": mean_wer,
        "eval/results_table": results_table,
    })

    wandb.finish()
    print("Evaluation complete.")


if __name__ == "__main__":
    with app.run():
        evaluate.remote()


## Inference

Load the fine-tuned model from the Modal volume and synthesize arbitrary
Swahili text. Audio is returned to the notebook for playback.


In [27]:
@app.function(
    image=image,
    volumes={"/checkpoints": vol},
)
def generate_audio(text: str, speaker_id: str = "6"):
    """Synthesize Swahili speech from text using the fine-tuned model.

    Args:
        text:       Swahili text to synthesize.
        speaker_id: Which speaker voice to clone. Must exist in WaxalNLP swa_tts.

    Returns:
        numpy array of audio samples at 16kHz.
    """
    import torch
    import numpy as np
    from datasets import load_dataset, Audio as DSAudio
    from speechbrain.inference.speaker import EncoderClassifier
    from transformers import (
        SpeechT5ForTextToSpeech,
        SpeechT5Processor,
        SpeechT5HifiGan,
    )

    C = CONFIG

    def clean_speech(wav, fade_samples=200):
        """Remove DC offset and apply fade in/out to prevent click artifacts."""
        wav = np.array(wav, dtype=np.float32)
        wav -= np.mean(wav)
        fade = np.linspace(0, 1, min(fade_samples, len(wav)))
        wav[:len(fade)] *= fade
        wav[-len(fade):] *= fade[::-1]
        return wav

    # -- Load models -----------------------------------------------------------
    print("Loading fine-tuned model from volume")
    processor = SpeechT5Processor.from_pretrained(C["base_checkpoint"])
    vocoder = SpeechT5HifiGan.from_pretrained(C["vocoder_checkpoint"])
    model = SpeechT5ForTextToSpeech.from_pretrained(C["final_model_dir"])
    model.eval()

    # -- Extract speaker embedding from reference audio ------------------------
    print("Extracting speaker embedding (speaker %s)" % speaker_id)
    ds = load_dataset(C["dataset"], C["dataset_config"], split="train", streaming=True)
    ds = ds.cast_column("audio", DSAudio(sampling_rate=C["sample_rate"]))
    sample = next(iter(ds.filter(lambda x: x["speaker_id"] == speaker_id)))

    spk_model = EncoderClassifier.from_hparams(
        source=C["speaker_encoder"],
        savedir="/tmp/speechbrain",
        run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    )
    with torch.no_grad():
        wav_tensor = torch.tensor(
            sample["audio"]["array"], dtype=torch.float32
        ).unsqueeze(0).to(spk_model.device)
        embedding = spk_model.encode_batch(wav_tensor)
        speaker_embedding = torch.nn.functional.normalize(embedding, dim=2)
        speaker_embedding = speaker_embedding.squeeze(0).cpu()

    # -- Synthesize ------------------------------------------------------------
    normalized = normalize_swahili_text(text)
    print("Synthesizing: '%s'" % normalized)
    inputs = processor(text=normalized, return_tensors="pt")

    with torch.no_grad():
        speech = model.generate_speech(
            inputs["input_ids"], speaker_embedding, vocoder=vocoder
        )

    return clean_speech(speech.numpy())


In [28]:
from IPython.display import Audio as IPythonAudio, display

if __name__ == "__main__":
    with app.run():
        test_phrases = [
            # Greeting — tests natural conversational prosody
            "Mambo vipi? Ninafuraha sana kuzungumza nawe leo.",
            # Statement — tests declarative intonation
            "Teknolojia ya akili bandia itabadilisha maisha yetu barani Afrika.",
            # Question — tests rising pitch at sentence end
            "Je, unafikiri tutaweza kujenga mifumo bora zaidi kesho?",
            # Short phrase — tests handling of brief utterances
            "Suala hilo, linapendeza sana.",
            # Numbers — tests the Swahili number normalizer
            "Kuna wanafunzi 1,500 katika shule hii.",
        ]

        for i, text in enumerate(test_phrases):
            print("\nGenerating phrase %d: '%s'" % (i + 1, text))
            audio_array = generate_audio.remote(text)
            print("Phrase %d ready (%.1f sec)" % (i + 1, len(audio_array) / 16000))
            display(IPythonAudio(audio_array, rate=16000))



Generating phrase 1: 'Mambo vipi? Ninafuraha sana kuzungumza nawe leo.'
Phrase 1 ready (7.0 sec)



Generating phrase 2: 'Teknolojia ya akili bandia itabadilisha maisha yetu barani Afrika.'
Phrase 2 ready (9.4 sec)



Generating phrase 3: 'Je, unafikiri tutaweza kujenga mifumo bora zaidi kesho?'
Phrase 3 ready (8.6 sec)



Generating phrase 4: 'Suala hilo, linapendeza sana.'
Phrase 4 ready (5.0 sec)



Generating phrase 5: 'Kuna wanafunzi 1,500 katika shule hii.'
Phrase 5 ready (6.2 sec)


## Setup notes

### Weights & Biases

The training function expects a Modal secret named `wandb-secret` containing
your `WANDB_API_KEY`. Create it once:

```bash
modal secret create wandb-secret WANDB_API_KEY=<your-key>
```

Runs will appear at: `https://wandb.ai/<your-username>/sauti-ya-kenya`

### Modal volume

Checkpoints persist across runs via the `sauti-tts-volume` Modal volume.
To inspect or download files:

```bash
modal volume ls sauti-tts-volume /checkpoints/
modal volume get sauti-tts-volume /checkpoints/test_v8_1.wav .
```


In [1]:
import modal
import os

app = modal.App("v10-msingi")
vol = modal.Volume.from_name("sauti-tts-volume")

@app.function(
    image=modal.Image.debian_slim().pip_install("datasets", "huggingface_hub", "tqdm"),
    volumes={"/mnt/sauti": vol},
    secrets=[modal.Secret.from_name("hf-secret")],
    timeout=3600
)
def initialize_v10_foundation():
    from datasets import load_dataset
    
    # --- LEVEL 1: STRUCTURE ---
    paths = [
        "/mnt/sauti/v10/data/raw/text",
        "/mnt/sauti/v10/data/raw/audio/phase1_foundation", # Target for Mozilla
        "/mnt/sauti/v10/data/raw/audio/phase2_stability",  # Target for CMU/Fleurs
        "/mnt/sauti/v10/data/processed/ss_gold",
        "/mnt/sauti/v10/msingi_tokens/checkpoints"
    ]
    for p in paths:
        os.makedirs(p, exist_ok=True)
    print("✅ Level 1: Structure Established.")

    # --- LEVEL 2: TEXT INGESTION (Msingi Foundation) ---
    token = os.environ.get("HF_TOKEN")
    text_dir = "/mnt/sauti/v10/data/raw/text"

    # 1. Wikipedia (The Grammar Anchor)
    print("⚓ Sinking Wikipedia Swahili...")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.sw", split="train", streaming=True)
    with open(f"{text_dir}/wiki_sw_fluent.txt", "w") as f:
        for i, ex in enumerate(wiki):
            f.write(ex["text"].replace("\n", " ") + "\n")
            if i >= 30000: break # Increased anchor size
            
    # 2. CulturaX (The Bulk Alpha)
    print("🚀 Sinking CulturaX Swahili (Bulk)...")
    try:
        bulk = load_dataset("uonlp/CulturaX", "sw", split="train", streaming=True, token=token)
        with open(f"{text_dir}/cc100_sw_bulk.txt", "w") as f:
            for i, ex in enumerate(bulk):
                f.write(ex["text"].replace("\n", " ") + "\n")
                if i % 10000 == 0: print(f"📦 Sunk {i} lines...")
                if i >= 500000: break
        print("✅ Level 2: Text Ingestion Complete.")
    except Exception as e:
        print(f"❌ Level 2 Failure: {e}")

    vol.commit()

if __name__ == "__main__":
    with app.run():
        initialize_v10_foundation.remote()

In [2]:
@app.function(
    image=modal.Image.debian_slim().pip_install("tqdm"),
    volumes={"/mnt/sauti": vol}
)
def run_v10_fluency_guard():
    import os
    import re
    from tqdm import tqdm

    # 1. Build the SS (Standard Swahili) Anchor
    anchor_path = "/mnt/sauti/v10/data/raw/text/wiki_sw_fluent.txt"
    ss_lexicon = set()
    
    print("⚓ Building Morphological Anchor from Wikipedia...")
    with open(anchor_path, "r", encoding="utf-8") as f:
        for line in f:
            words = re.findall(r'\b\w+\b', line.lower())
            ss_lexicon.update(words)
    print(f"Lexicon built: {len(ss_lexicon):,} unique standard words.")

    # 2. Filter the Bulk CulturaX
    bulk_path = "/mnt/sauti/v10/data/raw/text/cc100_sw_bulk.txt"
    gold_path = "/mnt/sauti/v10/data/processed/ss_gold/ss_gold_v10.txt"
    
    print(f"🧹 Guarding Fluency on {bulk_path}...")
    gold_count = 0

    with open(bulk_path, "r", encoding="utf-8") as fin, \
         open(gold_path, "w", encoding="utf-8") as fgold:
        
        for line in tqdm(fin):
            words = re.findall(r'\b\w+\b', line.lower())
            if not words: continue
            
            # Score: what % of words are in our 'Standard' anchor?
            matches = sum(1 for w in words if w in ss_lexicon)
            score = matches / len(words)
            
            # 80% threshold ensures we keep the 'Signal' and ditch the 'Noise'
            if score >= 0.80: 
                fgold.write(line)
                gold_count += 1
                
    print(f"✅ Level 4 Complete!")
    print(f"--- Gold Tier: {gold_count:,} lines (The 'Msingi' Foundation)")
    
    vol.commit()

if __name__ == "__main__":
    with app.run():
        run_v10_fluency_guard.remote()

In [17]:
@app.function(
    image=modal.Image.debian_slim().pip_install("requests", "tqdm"),
    volumes={"/mnt/sauti": vol},
    secrets=[modal.Secret.from_name("mozilla-secret")],
    timeout=7200 
)
def sink_v10_audio_foundation():
    import os
    import requests
    from tqdm import tqdm

    # EXACT credentials and IDs from your documentation
    API_KEY = os.environ["MDC_API_KEY"]
    DATASET_ID = "cmn3ailbd008nmb07mjyu3xro"
    API_URL = f"https://datacollective.mozillafoundation.org/api/datasets/{DATASET_ID}/download"
    
    dest_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation"
    os.makedirs(dest_path, exist_ok=True)

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    print(f"🚀 Initializing Phase 1 Sink for Dataset: {DATASET_ID}")

    try:
        # Step 1: POST to get the presigned URL
        print("🔗 Requesting presigned download session...")
        response = requests.post(API_URL, headers=headers)
        response.raise_for_status()
        
        data = response.json()
        download_url = data.get("downloadUrl")
        filename = data.get("filename", "swahili_mcv_foundation.tar.gz")

        if not download_url:
            print(f"❌ API Error: No downloadUrl in response. Body: {data}")
            return

        print(f"✅ Presigned URL generated. File: {filename}")

        # Step 2: Stream the .tar.gz (20.87 GB)
        print(f"📥 Streaming audio foundation to {dest_path}...")
        save_path = os.path.join(dest_path, filename)
        
        with requests.get(download_url, stream=True) as r:
            r.raise_for_status()
            # 20GB is massive; using a larger chunk size (1MB) for volume efficiency
            with open(save_path, 'wb') as f:
                for chunk in tqdm(r.iter_content(chunk_size=1024*1024), unit="MB", desc="Sinking"):
                    f.write(chunk)
                        
        print(f"🏁 Level 3 Phase 1 Complete! Audio foundation is on the disk.")

    except Exception as e:
        print(f"❌ MDC Pipeline Failure: {str(e)}")

    vol.commit()

if __name__ == "__main__":
    with app.run():
        sink_v10_audio_foundation.remote()

In [6]:
@app.function(volumes={"/mnt/sauti": vol})
def audit_msingi_vocab():
    import json
    import os

    vocab_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/vocab.json"
    
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    
    # Sort by token ID (lower IDs are usually the most frequent/earliest merges)
    sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
    
    print("💎 MsingiTokens: Top 50 Discovered Merges")
    print("-" * 40)
    for token, token_id in sorted_vocab[100:150]: # Skip the single-char basics
        print(f"ID {token_id:5} | Token: {token}")

if __name__ == "__main__":
    with app.run():
        audit_msingi_vocab.remote()

In [10]:
@app.function(volumes={"/mnt/sauti": vol})
def audit_swahili_merges():
    import json
    # Accessing the vocab we just saved in v10-msingi
    vocab_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/vocab.json"
    
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    
    # Sort and skip the 256 byte-tokens to find the Swahili patterns
    sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
    sw_units = sorted_vocab[256:306] 
    
    print("💎 MsingiTokens: Top 50 Agglutinative Merges")
    print("-" * 50)
    for token, token_id in sw_units:
        # 'Ġ' represents a space in Byte-Level BPE
        clean = token.replace('Ġ', ' ')
        print(f"ID {token_id:5} | unit: [{clean}]")

if __name__ == "__main__":
    with app.run():
        audit_swahili_merges.remote()

In [22]:
@app.function(
    volumes={"/mnt/sauti": vol},
    timeout=600 
)
def scout_v10_tarball():
    import os
    import tarfile

    base_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation"
    
    try:
        tarball = [f for f in os.listdir(base_path) if f.endswith(".tar.gz")][0]
        tar_path = os.path.join(base_path, tarball)
        print(f"📦 Scouting Blob: {tarball}")
    except Exception:
        print(f"❌ Error: No .tar.gz found in {base_path}.")
        return

    print("🔍 Listing all metadata/text files inside the archive...")
    with tarfile.open(tar_path, "r:gz") as tar:
        # We look for anything that isn't a .mp3 or .wav
        members = tar.getmembers()
        metadata_files = [
            m.name for m in members 
            if m.name.endswith(('.tsv', '.csv', '.json', '.txt')) 
            and "/clips/" not in m.name
        ]
        
        print(f"--- Found {len(metadata_files)} potential metadata files ---")
        for name in metadata_files:
            # We also peek at the size to see which one is the 'Master'
            info = tar.getmember(name)
            size_mb = info.size / (1024 * 1024)
            print(f"📄 {name:40} | Size: {size_mb:.2f} MB")

if __name__ == "__main__":
    with app.run():
        scout_v10_tarball.remote()

In [23]:
@app.function(
    image=modal.Image.debian_slim().pip_install("tokenizers", "pandas", "tqdm"),
    volumes={"/mnt/sauti": vol},
    timeout=7200 
)
def run_v10_extraction_v4():
    import os
    import tarfile
    import pandas as pd
    from tokenizers import Tokenizer
    from tqdm import tqdm
    import json

    base_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation"
    ckpt_dir = "/mnt/sauti/v10/msingi_tokens/checkpoints"
    tokenizer_json_path = os.path.join(ckpt_dir, "tokenizer.json")
    
    # 1. Load the Msingi-Brain
    tokenizer = Tokenizer.from_file(tokenizer_json_path)
    print("🧠 MsingiTokens Tokenizer Loaded.")

    # 2. Locate the Archive
    tarball = [f for f in os.listdir(base_path) if f.endswith(".tar.gz")][0]
    tar_path = os.path.join(base_path, tarball)
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest.jsonl"
    
    # Target file found in Scout
    TARGET_TSV = "cv-corpus-25.0-2026-03-09/sw/validated.tsv"

    print(f"🏗️  Extracting Master Metadata: {TARGET_TSV}...")
    with tarfile.open(tar_path, "r:gz") as tar:
        # Extract the specific validated.tsv
        member = tar.getmember(TARGET_TSV)
        tar.extract(member, path=base_path)
        
        # Load the data
        df = pd.read_csv(os.path.join(base_path, TARGET_TSV), sep='\t')
        print(f"✅ Metadata Loaded: {len(df):,} validated samples found.")

        # 3. Build the Msingi-Manifest
        print("📝 Mapping MsingiTokens to Acoustic Paths...")
        with open(manifest_path, "w", encoding="utf-8") as f_out:
            for _, row in tqdm(df.iterrows(), total=len(df)):
                text = str(row['sentence']).lower()
                token_ids = tokenizer.encode(text).ids
                
                # We point to the clips folder where audio will be extracted
                entry = {
                    "audio_path": f"{base_path}/cv-corpus-25.0-2026-03-09/sw/clips/{row['path']}",
                    "text": text,
                    "msingi_tokens": token_ids,
                    "up_votes": row.get('up_votes', 0),
                    "gender": row.get('gender', 'unknown')
                }
                f_out.write(json.dumps(entry) + "\n")

        # 4. Alpha Audio Verify
        print("🔊 Extracting first 1,000 clips for verification...")
        # Note the nested structure from the tarball
        prefix = "cv-corpus-25.0-2026-03-09/sw/clips/"
        audio_members = [m for m in tar.getmembers() if m.name.startswith(prefix)][:1000]
        tar.extractall(path=base_path, members=audio_members)

    print(f"🏁 Level 3.5 Complete! Manifest ready at {manifest_path}")
    vol.commit()

if __name__ == "__main__":
    with app.run():
        run_v10_extraction_v4.remote()

In [24]:
@app.function(volumes={"/mnt/sauti": vol})
def peek_v10_manifest():
    import json
    
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest.jsonl"
    
    print("📖 Peeking at the Msingi-Acoustic Bridge...")
    print("-" * 60)
    
    with open(manifest_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            data = json.loads(line)
            print(f"Sample {i+1}:")
            print(f"📝 Text: {data['text']}")
            print(f"🔢 Tokens: {data['msingi_tokens']}")
            print(f"🔊 Audio: {data['audio_path']}")
            print("-" * 30)
            if i >= 2: break

if __name__ == "__main__":
    with app.run():
        peek_v10_manifest.remote()

In [28]:
@app.function(
    image=modal.Image.debian_slim().pip_install("torch", "transformers", "librosa", "soundfile"),
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000",
    timeout=14400 
)
def run_v10_real_training():
    import torch
    import librosa
    import json
    from transformers import SpeechT5ForTextToSpeech, SpeechT5Config, SpeechT5Processor, SpeechT5FeatureExtractor
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest.jsonl"
    
    # 1. The Acoustic Gatekeeper
    # We use a feature extractor to turn raw audio into Log-Mel Spectrograms
    feature_extractor = SpeechT5FeatureExtractor() 
    
    # 2. Re-initialize Model
    config = SpeechT5Config(vocab_size=32000)
    model = SpeechT5ForTextToSpeech(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    model.train()

    print("🎙️ Starting Real Acoustic Training...")

    with open(manifest_path, "r") as f:
        for i, line in enumerate(f):
            sample = json.loads(line)
            
            # --- AUDIO LOADING ---
            try:
                audio_path = sample["audio_path"]
                # Load audio and resample to 16kHz (Standard for SpeechT5)
                audio, sr = librosa.load(audio_path, sr=16000)
                
                # Convert to Log-Mel Spectrogram (The 'Labels')
                labels = feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_values
                labels = labels.to(device)
                
                # --- TOKEN LOADING ---
                input_ids = torch.tensor([sample["msingi_tokens"]]).to(device)
                
                # --- FORWARD PASS (No more NoneType!) ---
                outputs = model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                
                # --- BACKPROP ---
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                
                if i % 10 == 0:
                    print(f"Batch {i} | Loss: {loss.item():.4f} | Tokens: {len(sample['msingi_tokens'])}")

            except Exception as e:
                # If audio is missing (since we only extracted 1,000 clips)
                continue
            
            if i >= 500: break # Alpha test limit

    # Save the first real 'Msingi' weights
    torch.save(model.state_dict(), "/mnt/sauti/v10/msingi_tokens/checkpoints/v10_foundation_v1.pt")
    print("🏁 Level 6.1 Complete! The model has officially 'heard' Swahili.")
    vol.commit()

if __name__ == "__main__":
    with app.run():
        run_v10_real_training.remote()

In [29]:
# Updated Image with OS-level dependencies for speed
training_image = (
    modal.Image.debian_slim()
    .apt_install("libsndfile1") # Fixes the PySoundFile warning
    .pip_install("torch", "transformers", "librosa", "soundfile", "matplotlib")
)

@app.function(
    image=training_image,
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000"
)
def msingi_v10_inference_test():
    import torch
    from transformers import SpeechT5ForTextToSpeech, SpeechT5Config
    from tokenizers import Tokenizer
    import matplotlib.pyplot as plt

    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/v10_foundation_v1.pt"
    tokenizer_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/tokenizer.json"

    # 1. Load the "Baby" v10 Model
    config = SpeechT5Config(vocab_size=32000)
    model = SpeechT5ForTextToSpeech(config).to(device)
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    # 2. Tokenize a "Click-Prone" test sentence
    tokenizer = Tokenizer.from_file(tokenizer_path)
    test_text = "wanamtaja mshambuliaji hodari"
    inputs = torch.tensor([tokenizer.encode(test_text).ids]).to(device)

    print(f"🧪 Testing Synthesis for: {test_text}")
    print(f"🔢 MsingiTokens: {tokenizer.encode(test_text).ids}")

    # 3. Generate Spectrogram
    with torch.no_grad():
        # Using a dummy speaker embedding for the foundation check
        speaker_embeddings = torch.zeros((1, 512)).to(device)
        outputs = model.generate_speech(inputs, speaker_embeddings)
    
    # 4. Visual Audit (Save spectrogram to volume)
    plt.figure(figsize=(10, 4))
    plt.imshow(outputs.cpu().numpy(), aspect='auto', origin='lower')
    plt.title(f"v10-msingi Zero-Shot: {test_text}")
    plt.savefig("/mnt/sauti/v10/data/processed/v10_alpha_spec.png")
    print("🎨 Spectrogram saved to /mnt/sauti/v10/data/processed/v10_alpha_spec.png")

if __name__ == "__main__":
    with app.run():
        msingi_v10_inference_test.remote()

In [30]:
@app.function(
    image=training_image,
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000",
    timeout=14400 # 4-hour blocks
)
def run_v10_foundation_full():
    import torch
    import librosa
    import json
    import os
    from transformers import SpeechT5ForTextToSpeech, SpeechT5Config, SpeechT5FeatureExtractor
    from tqdm import tqdm

    device = "cuda" if torch.cuda.is_available() else "cpu"
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest.jsonl"
    save_dir = "/mnt/sauti/v10/msingi_tokens/checkpoints"
    
    feature_extractor = SpeechT5FeatureExtractor()
    config = SpeechT5Config(vocab_size=32000)
    model = SpeechT5ForTextToSpeech(config).to(device)
    
    # Load previous Alpha weights if they exist to continue progress
    alpha_ckpt = os.path.join(save_dir, "v10_foundation_v1.pt")
    if os.path.exists(alpha_ckpt):
        model.load_state_dict(torch.load(alpha_ckpt))
        print("🔄 Resuming from Alpha weights...")

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5) # Lower LR for stability
    model.train()

    print(f"🌊 Starting the Great Sink: 267k Samples...")

    with open(manifest_path, "r") as f:
        for i, line in tqdm(enumerate(f)):
            sample = json.loads(line)
            
            try:
                # Load and process audio
                audio, _ = librosa.load(sample["audio_path"], sr=16000)
                labels = feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_values.to(device)
                input_ids = torch.tensor([sample["msingi_tokens"]]).to(device)
                
                # Step
                outputs = model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                loss.backward()
                
                # Gradient Accumulation (optional, but good for stability)
                if i % 4 == 0:
                    optimizer.step()
                    optimizer.zero_grad()
                
                # Checkpointing every 5000 samples
                if i % 5000 == 0 and i > 0:
                    ckpt_name = f"v10_foundation_step_{i}.pt"
                    torch.save(model.state_dict(), os.path.join(save_dir, ckpt_name))
                    print(f"💾 Checkpoint saved: {ckpt_name} | Loss: {loss.item():.4f}")
                    vol.commit()

            except Exception:
                continue # Skip missing clips from Alpha extraction

    torch.save(model.state_dict(), os.path.join(save_dir, "v10_foundation_final_epoch1.pt"))
    print("🏁 Phase 1 Foundation: Epoch 1 Complete.")
    vol.commit()

if __name__ == "__main__":
    with app.run():
        run_v10_foundation_full.remote()

In [40]:
@app.function(
    # We explicitly tell Modal to include tqdm in the environment for this function
    image=modal.Image.debian_slim().pip_install("tqdm"),
    volumes={"/mnt/sauti": vol}
)
def fix_manifest_order():
    import tarfile
    import json
    import os
    from tqdm import tqdm

    base_dir = "/mnt/sauti/v10/data/raw/audio/phase1_foundation"
    
    # Locate the tarball
    tar_files = [f for f in os.listdir(base_dir) if f.endswith(".tar.gz")]
    if not tar_files:
        print("❌ No tarball found!")
        return
        
    tar_path = os.path.join(base_dir, tar_files[0])
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest.jsonl"
    sorted_path = "/mnt/sauti/v10/data/processed/phase1_manifest_sorted.jsonl"

    print("📋 Indexing tarball physical order (The Bypass)...")
    with tarfile.open(tar_path, "r:gz") as tar:
        # Get names in the exact physical order they appear in the compressed stream
        # This is the secret sauce to O(N) speed
        tar_order = {m.name: i for i, m in enumerate(tar.getmembers())}

    print("🔄 Re-aligning manifest to tarball stream...")
    with open(manifest_path, "r") as f:
        samples = [json.loads(line) for line in f]

    # Sort based on the physical index in the tarball
    def get_index(sample):
        # Match the manifest path to the tar member name
        member_name = "/".join(sample["audio_path"].split("/")[-4:])
        return tar_order.get(member_name, 999999)

    samples.sort(key=get_index)

    print(f"📝 Writing {len(samples)} sorted entries...")
    with open(sorted_path, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")
            
    print(f"🏁 Sorted manifest saved to: {sorted_path}")
    vol.commit()

if __name__ == "__main__":
    with app.run():
        fix_manifest_order.remote()

In [66]:
import modal
import os

app = modal.App("sauti-ya-kenya-v10")
vol = modal.Volume.from_name("sauti-tts-volume")

# 1. Battle-Hardened Image
v10_final_image = (
    modal.Image.debian_slim()
    .apt_install("libsndfile1", "ffmpeg") 
    .pip_install(
        "torch", "transformers", "librosa", "soundfile", 
        "bigvgan", "wandb", "tqdm", "torchaudio", 
        "huggingface_hub", "auraloss", "pesq", "pystoi"
    )
)

# 2. GLOBAL HELPER (With explicit torch import)
def get_msingi_mel(wav, device):
    import torch
    import torchaudio
    wav = wav.to(device)
    # BigVGAN-v2 24kHz standard: 100 bands, 1024 n_fft, 256 hop
    transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=24000, n_fft=1024, win_length=1024,
        hop_length=256, f_min=0, f_max=12000, n_mels=100,
        center=False, power=1.0, normalized=False
    ).to(device)
    mel = transform(wav)
    return torch.log(torch.clamp(mel, min=1e-5))

class AttrDict(dict):
    def __getattr__(self, name): return self[name]

@app.function(
    image=v10_final_image,
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000", # The $18/hr beast
    memory=64000, 
    timeout=43200,
    secrets=[modal.Secret.from_name("wandb-secret"), modal.Secret.from_name("hf-secret")] 
)
def train_v10_final_sprint():
    import tarfile, io, json, os, torch, librosa, wandb, auraloss
    from bigvgan import BigVGAN
    from tqdm import tqdm
    from huggingface_hub import hf_hub_download
    from pesq import pesq
    from pystoi import stoi

    wandb.init(project="sauti-ya-kenya-v10", name="bigvgan-v2-final-fix-v10.3")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Assemble Model
    print("🏗️  Assembling BigVGAN-v2...")
    repo_id = "nvidia/bigvgan_v2_24khz_100band_256x"
    ckpt_path = hf_hub_download(repo_id=repo_id, filename="bigvgan_generator.pt")
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
    
    with open(config_path, "r") as f:
        h = AttrDict(json.load(f))
    
    model = BigVGAN(h).to(device)
    state_dict = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state_dict['generator'])
    model.train()

    stft_loss_fn = auraloss.freq.MultiResolutionSTFTLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # 2. RAM Load (The Speed King)
    tar_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation/common-voice-scripted-speech-25-0-swahil-dfb28a71.tar.gz"
    print("🚀 Sucking 20.87GB into RAM...")
    with open(tar_path, "rb") as f:
        ram_buffer = io.BytesIO(f.read())
    
    # 3. Lookup Index
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest_sorted.jsonl"
    validated_samples = {}
    with open(manifest_path, "r") as f:
        for line in f:
            s = json.loads(line)
            # Normalizing keys: common_voice_sw_123.mp3
            validated_samples[os.path.basename(s["audio_path"])] = s["msingi_tokens"]

    # 4. The Linear Stream
    print("🌊 Starting High-Speed Stream...")
    success_count = 0
    attempted_matches = 0
    
    with tarfile.open(fileobj=ram_buffer, mode="r:gz") as tar:
        pbar = tqdm(total=len(validated_samples), desc="v10 training")
        
        for member in tar:
            if not member.isfile(): continue
            
            member_key = os.path.basename(member.name)
            
            if member_key in validated_samples:
                attempted_matches += 1
                try:
                    # Robust loading with librosa
                    audio_bytes = tar.extractfile(member).read()
                    audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=24000)
                    
                    # Convert to tensor (This is where the 'torch' error was!)
                    wav = torch.FloatTensor(audio).unsqueeze(0).to(device)
                    
                    # --- THE REAL TRAINING STEP ---
                    mel = get_msingi_mel(wav, device)
                    y_hat = model(mel)
                    
                    t_len = min(y_hat.shape[-1], wav.shape[-1])
                    loss = stft_loss_fn(y_hat[:, :, :t_len], wav[:, :t_len])
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                    success_count += 1
                    pbar.update(1)

                    if success_count % 10 == 0:
                        wandb.log({"STFT_Loss": loss.item(), "step": success_count})

                except Exception as e:
                    print(f"⚠️ Runtime Error on {member_key}: {e}")
                    continue

            # NEW KILL SWITCH: Only kills if we FOUND files but they CRASHED 10 times in a row.
            if attempted_matches > 10 and success_count == 0:
                raise RuntimeError("Code is crashing on valid files. Check your logic!")

    vol.commit()

if __name__ == "__main__":
    with app.run():
        train_v10_final_sprint.remote()

In [2]:
import os
import torch

# Inside your function, right after model assembly:
checkpoint_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/v10_vocoder_25500.pt"
if os.path.exists(checkpoint_path):
    print(f"♻️ Resuming from checkpoint: {checkpoint_path}")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    # Optional: adjust your success_count to match the checkpoint
    success_count = 25500 
else:
    print("🆕 No checkpoint found, starting from foundation weights.")

@app.function(
    image=v10_final_image,
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000", 
    memory=128000, 
    timeout=43200, # 1. INCREASED TO 12 HOURS (To ensure it finishes the night)
    secrets=[modal.Secret.from_name("wandb-secret"), modal.Secret.from_name("hf-secret")] 
)
def train_v10_final_sprint():
    import tarfile, io, json, os, torch, librosa, wandb, auraloss
    from bigvgan import BigVGAN
    from tqdm import tqdm
    from huggingface_hub import hf_hub_download

    wandb.init(project="sauti-ya-kenya-v10", name="v13-the-sleep-script")
    device = "cuda"
    
    # Assembly
    print("🏗️  Assembling BigVGAN-v2...")
    repo_id = "nvidia/bigvgan_v2_24khz_100band_256x"
    ckpt_path = hf_hub_download(repo_id=repo_id, filename="bigvgan_generator.pt")
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
    with open(config_path, "r") as f:
        h = AttrDict(json.load(f))
    model = BigVGAN(h).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu')['generator'])
    model.train()

    stft_loss_fn = auraloss.freq.MultiResolutionSTFTLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # 2. RAM Load
    tar_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation/common-voice-scripted-speech-25-0-swahil-dfb28a71.tar.gz"
    print("🚀 Sucking 20.87GB into RAM...")
    with open(tar_path, "rb") as f:
        ram_buffer = io.BytesIO(f.read())
    
    # 3. Index
    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest_sorted.jsonl"
    validated_samples = {}
    with open(manifest_path, "r") as f:
        for line in f:
            s = json.loads(line)
            validated_samples[os.path.basename(s["audio_path"])] = s["msingi_tokens"]

    # 4. The Linear Stream (YOUR WORKING LOGIC)
    print("🌊 Starting High-Speed Stream...")
    success_count = 0
    
    with tarfile.open(fileobj=ram_buffer, mode="r:gz") as tar:
        pbar = tqdm(total=len(validated_samples), desc="v10 Training")
        for member in tar:
            if not member.isfile(): continue
            member_key = os.path.basename(member.name)
            
            if member_key in validated_samples:
                try:
                    audio_bytes = tar.extractfile(member).read()
                    audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=24000)
                    
                    wav = torch.FloatTensor(audio).unsqueeze(0).to(device)
                    mel = get_msingi_mel(wav, device)
                    y_hat = model(mel)
                    
                    t_len = min(y_hat.shape[-1], wav.shape[-1])
                    loss = stft_loss_fn(y_hat[:, :, :t_len], wav[:, :t_len])
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                    success_count += 1
                    pbar.update(1)

                    if success_count % 10 == 0:
                        wandb.log({"STFT_Loss": loss.item(), "step": success_count})

                    # 2. ADDED FREQUENT CHECKPOINTING (Every 500 steps)
                    if success_count % 500 == 0:
                        save_path = f"/mnt/sauti/v10/msingi_tokens/checkpoints/v10_vocoder_{success_count}.pt"
                        torch.save(model.state_dict(), save_path)
                        vol.commit() # Burns progress into the volume

                except Exception:
                    continue

    vol.commit()
    wandb.finish()

if __name__ == "__main__":
    with app.run():
        train_v10_final_sprint.remote()

🆕 No checkpoint found, starting from foundation weights.


NameError: name 'app' is not defined

In [1]:
%%writefile train_msingi_v10_gan.py


import modal
import os

# 1. INFRASTRUCTURE SETUP
app = modal.App("msingi-v10-gan-pivot")
vol = modal.Volume.from_name("sauti-tts-volume")

v10_gan_image = (
    modal.Image.debian_slim()
    .apt_install("libsndfile1", "ffmpeg", "ninja-build", "gcc", "g++")
    .pip_install(
        "torch", "transformers", "librosa", "soundfile", 
        "bigvgan", "wandb", "tqdm", "torchaudio", 
        "huggingface_hub", "auraloss", "hf_transfer"
    )
)

# --- THE DISCRIMINATOR ARCHITECTURES (LOCAL DEFINITIONS) ---
import torch
import torch.nn as nn
from torch.nn.utils import weight_norm, spectral_norm

class DiscriminatorP(nn.Module):
    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super(DiscriminatorP, self).__init__()
        self.period = period
        norm_f = weight_norm if not use_spectral_norm else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(2, 0))),
        ])
        self.conv_post = norm_f(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []
        # Reshape for periodic convolution
        b, c, t = x.shape
        if t % self.period != 0:
            n_pad = self.period - (t % self.period)
            x = torch.nn.functional.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for l in self.convs:
            x = l(x)
            x = torch.nn.functional.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)
        return x, fmap

class MultiPeriodDiscriminator(nn.Module):
    def __init__(self, h):
        super(MultiPeriodDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorP(2), DiscriminatorP(3), DiscriminatorP(5), DiscriminatorP(7), DiscriminatorP(11)
        ])
    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        for i, d in enumerate(self.discriminators):
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r); fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g); fmap_gs.append(fmap_g)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

class DiscriminatorS(nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(DiscriminatorS, self).__init__()
        norm_f = weight_norm if not use_spectral_norm else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv1d(1, 128, 15, 1, padding=7)),
            norm_f(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            norm_f(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            norm_f(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = l(x)
            x = torch.nn.functional.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)
        return x, fmap

class MultiScaleDiscriminator(nn.Module):
    def __init__(self, h):
        super(MultiScaleDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorS(use_spectral_norm=True), DiscriminatorS(), DiscriminatorS()
        ])
        self.meanpools = nn.ModuleList([nn.AvgPool1d(4, 2, padding=2), nn.AvgPool1d(4, 2, padding=2)])
    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        for i, d in enumerate(self.discriminators):
            if i != 0:
                y = self.meanpools[i-1](y)
                y_hat = self.meanpools[i-1](y_hat)
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r); fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g); fmap_gs.append(fmap_g)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

# --- END ARCHITECTURES ---

class AttrDict(dict):
    def __getattr__(self, name): return self[name]

def get_msingi_mel(wav, device):
    import torch, torchaudio
    transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=24000, n_fft=1024, win_length=1024,
        hop_length=256, f_min=0, f_max=12000, n_mels=100,
        center=False, power=1.0, normalized=False
    ).to(device)
    mel = transform(wav)
    return torch.log(torch.clamp(mel, min=1e-5))

@app.function(
    image=v10_gan_image,
    volumes={"/mnt/sauti": vol},
    gpu="RTX-PRO-6000", 
    memory=128000, 
    timeout=64800, 
    secrets=[modal.Secret.from_name("wandb-secret"), modal.Secret.from_name("hf-secret")] 
)
def train_v10_gan_sprint():
    import tarfile, io, json, os, torch, librosa, wandb, auraloss
    from bigvgan import BigVGAN
    from tqdm import tqdm
    from huggingface_hub import hf_hub_download

    device = "cuda"
    wandb.init(project="sauti-ya-kenya-v10", name="v10.5-gan-stabilization")

    repo_id = "nvidia/bigvgan_v2_24khz_100band_256x"
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
    with open(config_path, "r") as f:
        h = AttrDict(json.load(f))
    
    generator = BigVGAN(h).to(device)
    mpd = MultiPeriodDiscriminator(h).to(device)
    msd = MultiScaleDiscriminator(h).to(device)

    checkpoint_path = "/mnt/sauti/v10/msingi_tokens/checkpoints/v10_vocoder_64000.pt"
    print(f"♻️  Resuming Generator from {checkpoint_path}...")
    generator.load_state_dict(torch.load(checkpoint_path, map_location=device))
    
    optim_g = torch.optim.AdamW(generator.parameters(), lr=1e-4, betas=(0.8, 0.99))
    optim_d = torch.optim.AdamW(list(mpd.parameters()) + list(msd.parameters()), lr=1e-4, betas=(0.8, 0.99))
    stft_loss_fn = auraloss.freq.MultiResolutionSTFTLoss().to(device)

    tar_path = "/mnt/sauti/v10/data/raw/audio/phase1_foundation/common-voice-scripted-speech-25-0-swahil-dfb28a71.tar.gz"
    with open(tar_path, "rb") as f:
        ram_buffer = io.BytesIO(f.read())

    manifest_path = "/mnt/sauti/v10/data/processed/phase1_manifest_sorted.jsonl"
    validated_samples = {}
    with open(manifest_path, "r") as f:
        for line in f:
            s = json.loads(line)
            validated_samples[os.path.basename(s["audio_path"])] = True

    print("🌊 Starting the Full GAN Sprint...")
    success_count = 64000
    with tarfile.open(fileobj=ram_buffer, mode="r:gz") as tar:
        pbar = tqdm(total=len(validated_samples), initial=64000, desc="GAN Sprint")
        for member in tar:
            if not member.isfile() or os.path.basename(member.name) not in validated_samples:
                continue
            try:
                audio_bytes = tar.extractfile(member).read()
                audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=24000)
                if len(audio) < 24000: continue
                audio = audio[:24000] 
                y = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)
                mel = get_msingi_mel(y.squeeze(1), device)

                # D Step
                optim_d.zero_grad()
                y_g_hat = generator(mel)
                y_df_hat_r, y_df_hat_g, _, _ = mpd(y, y_g_hat.detach())
                loss_d = sum([torch.mean((dr-1)**2) + torch.mean(dg**2) for dr, dg in zip(y_df_hat_r, y_df_hat_g)])
                y_ds_hat_r, y_ds_hat_g, _, _ = msd(y, y_g_hat.detach())
                loss_d += sum([torch.mean((dr-1)**2) + torch.mean(dg**2) for dr, dg in zip(y_ds_hat_r, y_ds_hat_g)])
                loss_d.backward(); optim_d.step()

                # G Step
                optim_g.zero_grad()
                loss_stft = stft_loss_fn(y_g_hat, y)
                _, y_df_hat_g, fmap_f_r, fmap_f_g = mpd(y, y_g_hat)
                _, y_ds_hat_g, fmap_s_r, fmap_s_g = msd(y, y_g_hat)
                loss_adv = sum([torch.mean((dg-1)**2) for dg in y_df_hat_g + y_ds_hat_g])
                loss_fm = sum([torch.mean(torch.abs(r - g)) for dr, dg in zip(fmap_f_r + fmap_s_r, fmap_f_g + fmap_s_g) for r, g in zip(dr, dg)])
                loss_g = (loss_stft * 45.0) + loss_adv + (loss_fm * 2.0)
                loss_g.backward(); optim_g.step()

                success_count += 1; pbar.update(1)
                if success_count % 50 == 0:
                    wandb.log({"G_Loss": loss_g.item(), "D_Loss": loss_d.item(), "STFT": loss_stft.item(), "step": success_count})
                if success_count % 1000 == 0:
                    torch.save(generator.state_dict(), f"/mnt/sauti/v10/msingi_tokens/checkpoints/v10_vocoder_{success_count}.pt")
                    vol.commit()
            except Exception: continue
    vol.commit(); wandb.finish()

if __name__ == "__main__":
    with app.run():
        train_v10_gan_sprint.remote()

Writing train_msingi_v10_gan.py
